# Building Segmentation Inference + PV Estimation — Kaggle

This is the **third operational notebook** in the pipeline.

```text
new sr.tif
    ↓
trained U-Net
    ↓
building probability map
    ↓
building_mask.tif
    ↓
building polygons
    ↓
roof area
    ↓
PV capacity
    ↓
tilt / azimuth optimization
    ↓
simple shading model
    ↓
annual clear-sky energy estimate
```

## Required Kaggle inputs

Attach datasets containing:

1. `sr.tif` — a new super-resolved GeoTIFF produced by notebook 1.
2. `unet_buildings.keras` — trained in notebook 2.
3. `segmentation_training_config.json` — saved with the trained model.

The notebook automatically searches `/kaggle/input` for these files.

## Important

The PV calculation preserves the main assumptions from your original `pv_estimation.ipynb`:

- 70% usable roof fraction;
- 200 W/m² PV power density;
- 14% generic system losses;
- clear-sky irradiance from `pvlib`;
- simple parametric surrounding-building shading;
- by default, the **largest detected building** is analyzed.

These results are therefore an **engineering estimate**, not a bankable PV yield study.

In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================

!pip install -q pvlib scikit-image

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

import rasterio
from rasterio import features
from rasterio.features import rasterize

import geopandas as gpd

from shapely.geometry import shape, Point
from shapely.prepared import prep

from pyproj import Transformer

from skimage.morphology import (
    remove_small_objects,
    remove_small_holes
)

import pvlib

print("TensorFlow:", tf.__version__)
print("Rasterio:", rasterio.__version__)
print("GeoPandas:", gpd.__version__)
print("pvlib:", pvlib.__version__)

print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Auto-detect the model, configuration, and new SR image

In [ ]:
# ============================================================
# 2. FIND INPUT FILES AUTOMATICALLY
# ============================================================

INPUT_ROOT = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/pv_pipeline_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------
# Find trained model
# -------------------------

model_matches = sorted(
    glob.glob(
        os.path.join(
            INPUT_ROOT,
            "**",
            "unet_buildings.keras"
        ),
        recursive=True
    )
)

if not model_matches:
    raise FileNotFoundError(
        "unet_buildings.keras was not found under /kaggle/input."
    )

MODEL_PATH = model_matches[0]


# -------------------------
# Find training config
# -------------------------

config_matches = sorted(
    glob.glob(
        os.path.join(
            INPUT_ROOT,
            "**",
            "segmentation_training_config.json"
        ),
        recursive=True
    )
)

if not config_matches:
    raise FileNotFoundError(
        "segmentation_training_config.json was not found under /kaggle/input."
    )

CONFIG_PATH = config_matches[0]


# -------------------------
# Find SR TIFF candidates
# -------------------------

all_tifs = sorted(
    glob.glob(
        os.path.join(
            INPUT_ROOT,
            "**",
            "*.tif"
        ),
        recursive=True
    )
)

# Prefer a file literally called sr.tif.
sr_exact = [
    p for p in all_tifs
    if os.path.basename(p).lower() == "sr.tif"
]

if sr_exact:
    IMAGE_TIF = sr_exact[0]
else:
    # Otherwise exclude known masks/probability outputs and take a TIFF
    # outside an obvious training dataset when possible.
    candidates = [
        p for p in all_tifs
        if "mask" not in os.path.basename(p).lower()
        and "prob" not in os.path.basename(p).lower()
    ]

    if not candidates:
        raise FileNotFoundError(
            "No suitable SR GeoTIFF was found under /kaggle/input."
        )

    IMAGE_TIF = candidates[0]


print("✅ Model:")
print(MODEL_PATH)

print("\\n✅ Config:")
print(CONFIG_PATH)

print("\\n✅ New SR image:")
print(IMAGE_TIF)

print("\\n✅ Output folder:")
print(OUTPUT_DIR)

## 3. Load the training configuration

Inference must reproduce the same band selection, normalization, patch size, and probability threshold used during training.

In [ ]:
# ============================================================
# 3. READ TRAINING CONFIG
# ============================================================

with open(CONFIG_PATH, "r") as f:
    train_config = json.load(f)

BANDS = train_config.get(
    "input_bands_1_based",
    [1, 2, 3]
)

PATCH_SIZE = int(
    train_config.get(
        "patch_size",
        256
    )
)

STRIDE = int(
    train_config.get(
        "recommended_inference_stride",
        PATCH_SIZE // 2
    )
)

PRED_THRESHOLD = float(
    train_config.get(
        "prediction_threshold",
        0.5
    )
)

norm_cfg = train_config.get(
    "normalization",
    {}
)

if isinstance(norm_cfg, dict):
    P_MIN = float(norm_cfg.get("pmin", 2))
    P_MAX = float(norm_cfg.get("pmax", 98))
else:
    # Backward compatibility with earlier config wording.
    P_MIN = 2.0
    P_MAX = 98.0

print("Bands       :", BANDS)
print("Patch size  :", PATCH_SIZE)
print("Stride      :", STRIDE)
print("Threshold   :", PRED_THRESHOLD)
print("Normalization percentiles:", P_MIN, P_MAX)

## 4. Load the trained U-Net and the new OpenSR image

In [ ]:
# ============================================================
# 4. LOAD MODEL + SR IMAGE
# ============================================================

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("✅ U-Net loaded")


def percentile_normalize(
    img,
    pmin=2,
    pmax=98,
    eps=1e-6
):
    img = img.astype(np.float32)

    out = np.zeros_like(
        img,
        dtype=np.float32
    )

    for c in range(img.shape[2]):

        band = img[..., c]

        finite = np.isfinite(band)

        if not finite.any():
            continue

        lo = np.percentile(
            band[finite],
            pmin
        )

        hi = np.percentile(
            band[finite],
            pmax
        )

        if hi - lo < eps:
            out[..., c] = 0.0
        else:
            out[..., c] = (
                band - lo
            ) / (
                hi - lo
            )

    out = np.nan_to_num(
        out,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    return np.clip(
        out,
        0.0,
        1.0
    )


with rasterio.open(IMAGE_TIF) as src:

    if src.count < max(BANDS):
        raise ValueError(
            f"SR image has {src.count} band(s), "
            f"but inference requires bands {BANDS}."
        )

    raw_rgb = np.stack(
        [
            src.read(b)
            for b in BANDS
        ],
        axis=-1
    )

    img_profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    bounds = src.bounds
    H = src.height
    W = src.width

img_rgb = percentile_normalize(
    raw_rgb,
    pmin=P_MIN,
    pmax=P_MAX
)

print("Image shape :", img_rgb.shape)
print("CRS         :", crs)
print("Resolution  :", (abs(transform.a), abs(transform.e)))
print("Bounds      :", bounds)

plt.figure(figsize=(8, 8))
plt.imshow(img_rgb)
plt.title("New OpenSR image")
plt.axis("off")
plt.show()

## 5. Tiled segmentation inference

The image is predicted in overlapping tiles. Overlapping probability predictions are averaged, which reduces tile-edge artifacts.

In [ ]:
# ============================================================
# 5. TILED U-NET INFERENCE
# ============================================================

def compute_starts(
    length,
    tile,
    stride
):
    if length <= tile:
        return [0]

    starts = list(
        range(
            0,
            length - tile + 1,
            stride
        )
    )

    last = length - tile

    if starts[-1] != last:
        starts.append(last)

    return starts


def predict_full_image(
    model,
    image,
    tile,
    stride
):
    H, W, C = image.shape

    # Pad if an image is smaller than one model tile.
    pad_h = max(0, tile - H)
    pad_w = max(0, tile - W)

    image_pad = np.pad(
        image,
        (
            (0, pad_h),
            (0, pad_w),
            (0, 0)
        ),
        mode="reflect"
    )

    Hp, Wp, _ = image_pad.shape

    prob_sum = np.zeros(
        (Hp, Wp),
        dtype=np.float32
    )

    weight_sum = np.zeros(
        (Hp, Wp),
        dtype=np.float32
    )

    y_starts = compute_starts(
        Hp,
        tile,
        stride
    )

    x_starts = compute_starts(
        Wp,
        tile,
        stride
    )

    total = len(y_starts) * len(x_starts)
    done = 0

    for y0 in y_starts:
        for x0 in x_starts:

            patch = image_pad[
                y0:y0 + tile,
                x0:x0 + tile
            ]

            pred = model.predict(
                patch[None, ...],
                verbose=0
            )[0, ..., 0]

            prob_sum[
                y0:y0 + tile,
                x0:x0 + tile
            ] += pred

            weight_sum[
                y0:y0 + tile,
                x0:x0 + tile
            ] += 1.0

            done += 1

            if done % 5 == 0 or done == total:
                print(
                    f"Tiles: {done}/{total}",
                    end="\r"
                )

    prob = (
        prob_sum
        / np.maximum(
            weight_sum,
            1e-8
        )
    )

    return prob[:H, :W]


prob_map = predict_full_image(
    model,
    img_rgb,
    tile=PATCH_SIZE,
    stride=STRIDE
)

print("\n✅ Inference complete")
print(
    "Probability range:",
    float(prob_map.min()),
    "→",
    float(prob_map.max())
)

plt.figure(figsize=(8, 8))
plt.imshow(
    prob_map,
    cmap="viridis",
    vmin=0,
    vmax=1
)
plt.colorbar(label="Building probability")
plt.title("U-Net building probability")
plt.axis("off")
plt.show()

## 6. Convert the probability map to a building mask

The model-specific threshold from training is applied. A light morphological cleanup removes tiny isolated objects and tiny holes.

These cleanup parameters are deliberately conservative and can be improved later with the segmentation model.

In [ ]:
# ============================================================
# 6. BINARY MASK + LIGHT POST-PROCESSING
# ============================================================

# Conservative cleanup for a 2.5 m/pixel OpenSR image.
MIN_OBJECT_PIXELS = 8
MAX_HOLE_PIXELS = 8

pred_mask_raw = (
    prob_map >= PRED_THRESHOLD
)

pred_mask = remove_small_objects(
    pred_mask_raw,
    min_size=MIN_OBJECT_PIXELS
)

pred_mask = remove_small_holes(
    pred_mask,
    area_threshold=MAX_HOLE_PIXELS
)

pred_mask = pred_mask.astype(
    np.uint8
)

print(
    f"Threshold: {PRED_THRESHOLD:.3f}"
)

print(
    "Raw building pixels:",
    int(pred_mask_raw.sum())
)

print(
    "Clean building pixels:",
    int(pred_mask.sum())
)


fig, axes = plt.subplots(
    1,
    4,
    figsize=(22, 6)
)

axes[0].imshow(img_rgb)
axes[0].set_title("SR RGB")

axes[1].imshow(
    prob_map,
    cmap="viridis",
    vmin=0,
    vmax=1
)
axes[1].set_title("Probability")

axes[2].imshow(
    pred_mask,
    cmap="gray"
)
axes[2].set_title("Building mask")

axes[3].imshow(img_rgb)
axes[3].imshow(
    pred_mask,
    cmap="Reds",
    alpha=0.40
)
axes[3].set_title("Segmentation overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Save the georeferenced probability and building-mask GeoTIFFs

In [ ]:
# ============================================================
# 7. SAVE SEGMENTATION OUTPUTS
# ============================================================

MASK_TIF = os.path.join(
    OUTPUT_DIR,
    "building_mask.tif"
)

PROB_TIF = os.path.join(
    OUTPUT_DIR,
    "building_probability.tif"
)


mask_profile = img_profile.copy()

mask_profile.update(
    count=1,
    dtype="uint8",
    nodata=0,
    compress="deflate"
)

with rasterio.open(
    MASK_TIF,
    "w",
    **mask_profile
) as dst:

    dst.write(
        pred_mask,
        1
    )


prob_profile = img_profile.copy()

prob_profile.update(
    count=1,
    dtype="float32",
    nodata=None,
    compress="deflate"
)

with rasterio.open(
    PROB_TIF,
    "w",
    **prob_profile
) as dst:

    dst.write(
        prob_map.astype(np.float32),
        1
    )


print("✅ Mask:")
print(MASK_TIF)

print("\n✅ Probability:")
print(PROB_TIF)

# PV Estimation

The remainder follows the structure of your original PV notebook:

1. vectorize detected buildings;
2. compute areas in a local projected CRS;
3. choose the largest detected building by default;
4. derive a usable roof area and DC capacity;
5. generate a full-year clear-sky solar resource;
6. optimize tilt and azimuth;
7. apply the simple surrounding-building shading approximation;
8. export results.

In [ ]:
# ============================================================
# 8. PV ASSUMPTIONS
# ============================================================

ROOF_USABLE_FRACTION = 0.70

MODULE_EFF = 0.20

POWER_DENSITY_W_PER_M2 = 200

SYSTEM_LOSS_FRACTION = 0.14

GAMMA_PDC = -0.0035

INV_EFF_NOM = 0.96

TZ = "UTC"

ALTITUDE_M = 0

PV_YEAR = 2025

TILT_GRID = list(
    range(
        0,
        46,
        5
    )
)

AZI_GRID = list(
    range(
        0,
        360,
        10
    )
)

USE_SHADING = True

DEFAULT_BUILDING_HEIGHT_M = 10.0

HEIGHT_BY_AREA = True

HEIGHT_MIN_M = 6.0
HEIGHT_MAX_M = 25.0

print("PV assumptions configured.")

## 9. Vectorize the predicted building mask

In [ ]:
# ============================================================
# 9. MASK → BUILDING POLYGONS
# ============================================================

shapes_gen = features.shapes(
    pred_mask,
    mask=(pred_mask == 1),
    transform=transform
)

geoms = [
    shape(g)
    for g, value in shapes_gen
    if int(value) == 1
]

if len(geoms) == 0:
    raise RuntimeError(
        "No building polygons were detected. "
        "Inspect the segmentation probability map and threshold."
    )

gdf = gpd.GeoDataFrame(
    {"geometry": geoms},
    crs=crs
).reset_index(drop=True)

# Remove invalid/empty polygons.
gdf = gdf[
    gdf.geometry.notnull()
    & ~gdf.geometry.is_empty
    & gdf.geometry.is_valid
].copy()

if len(gdf) == 0:
    raise RuntimeError(
        "All vectorized building geometries were invalid."
    )

utm_crs = gdf.estimate_utm_crs()

if utm_crs is None:
    raise RuntimeError(
        "Could not estimate a local projected CRS."
    )

gdf_utm = gdf.to_crs(
    utm_crs
)

gdf["area_m2"] = (
    gdf_utm.geometry.area.values
)

# Remove extremely small segmentation fragments in physical units.
MIN_BUILDING_AREA_M2 = 20.0

keep = (
    gdf["area_m2"]
    >= MIN_BUILDING_AREA_M2
)

gdf = gdf.loc[
    keep
].copy().reset_index(drop=True)

if len(gdf) == 0:
    raise RuntimeError(
        "No detected polygons remain after the minimum-area filter."
    )

gdf["building_id"] = np.arange(
    1,
    len(gdf) + 1
)

# Reproject filtered polygons again for consistent row order.
gdf_utm = gdf.to_crs(
    utm_crs
)

# Calculate centroids in the projected CRS, then transform to WGS84.
centroids_utm = (
    gdf_utm.geometry.centroid
)

centroid_gdf = gpd.GeoDataFrame(
    geometry=centroids_utm,
    crs=utm_crs
).to_crs(
    "EPSG:4326"
)

gdf["lon"] = (
    centroid_gdf.geometry.x.values
)

gdf["lat"] = (
    centroid_gdf.geometry.y.values
)

print("Detected buildings:", len(gdf))

display(
    gdf[
        [
            "building_id",
            "area_m2",
            "lat",
            "lon"
        ]
    ]
    .sort_values(
        "area_m2",
        ascending=False
    )
    .head(20)
)

## 10. Select the building for the PV study

Default behavior is intentionally the same as your original notebook: analyze the **largest detected roof/building footprint**.

In [ ]:
# ============================================================
# 10. SELECT LARGEST DETECTED BUILDING
# ============================================================

b = (
    gdf
    .sort_values(
        "area_m2",
        ascending=False
    )
    .iloc[0]
)

BID = int(
    b["building_id"]
)

roof_poly = (
    gdf.loc[
        gdf["building_id"] == BID,
        "geometry"
    ]
    .iloc[0]
)

neighbors = gdf[
    gdf["building_id"] != BID
].copy()

print("Selected building ID:", BID)
print(
    f"Detected roof/footprint area: "
    f"{float(b['area_m2']):.2f} m²"
)
print(
    "Lat/lon:",
    (
        float(b["lat"]),
        float(b["lon"])
    )
)

sel_mask = rasterize(
    [(roof_poly, 1)],
    out_shape=(H, W),
    transform=transform,
    fill=0,
    dtype=np.uint8
)

plt.figure(
    figsize=(8, 8)
)

plt.imshow(img_rgb)

plt.imshow(
    sel_mask,
    cmap="Greens",
    alpha=0.5
)

plt.title(
    f"Selected building #{BID}"
)

plt.axis("off")
plt.show()

## 11. Clear-sky solar resource for the building location

In [ ]:
# ============================================================
# 11. SOLAR RESOURCE
# ============================================================

lat = float(
    b["lat"]
)

lon = float(
    b["lon"]
)

times = pd.date_range(
    f"{PV_YEAR}-01-01",
    f"{PV_YEAR}-12-31 23:00:00",
    freq="1h",
    tz=TZ
)

location = pvlib.location.Location(
    latitude=lat,
    longitude=lon,
    tz=TZ,
    altitude=ALTITUDE_M
)

cs = location.get_clearsky(
    times,
    model="ineichen"
)

solpos = location.get_solarposition(
    times
)

print(
    f"Solar resource generated for "
    f"{len(times)} hourly timestamps."
)

display(cs.head())

## 12. PV power model

In [ ]:
# ============================================================
# 12. PV MODEL FUNCTIONS
# ============================================================

def poa_irradiance(
    surface_tilt,
    surface_azimuth,
    solpos,
    cs
):
    return pvlib.irradiance.get_total_irradiance(

        surface_tilt=surface_tilt,

        surface_azimuth=surface_azimuth,

        solar_zenith=solpos[
            "apparent_zenith"
        ],

        solar_azimuth=solpos[
            "azimuth"
        ],

        dni=cs["dni"],

        ghi=cs["ghi"],

        dhi=cs["dhi"],

        dni_extra=pvlib.irradiance.get_extra_radiation(
            solpos.index
        ),

        model="haydavies"
    )


def cell_temperature(
    poa_global,
    temp_air=20.0,
    wind_speed=1.0
):
    params = (
        pvlib.temperature
        .TEMPERATURE_MODEL_PARAMETERS[
            "sapm"
        ][
            "open_rack_glass_glass"
        ]
    )

    return pvlib.temperature.sapm_cell(
        poa_global,
        temp_air,
        wind_speed,
        **params
    )


def pvwatts_power(
    poa_effective,
    tcell,
    pdc0_w,
    gamma_pdc=GAMMA_PDC,
    inv_eff=INV_EFF_NOM
):
    pdc = pvlib.pvsystem.pvwatts_dc(
        poa_effective,
        tcell,
        pdc0=pdc0_w,
        gamma_pdc=gamma_pdc
    )

    pac = pvlib.inverter.pvwatts(
        pdc,
        pdc0=pdc0_w,
        eta_inv_nom=inv_eff
    )

    return pdc, pac

## 13. Simple surrounding-building shading model

This is preserved as a **parametric approximation** from your original notebook. Building heights are inferred crudely from footprint area because the segmentation image itself does not provide height.

In [ ]:
# ============================================================
# 13. SHADING FUNCTIONS
# ============================================================

def building_height_m(
    area_m2
):
    if not HEIGHT_BY_AREA:
        return DEFAULT_BUILDING_HEIGHT_M

    h = (
        HEIGHT_MIN_M
        + (
            HEIGHT_MAX_M
            - HEIGHT_MIN_M
        )
        * (
            area_m2
            / (
                area_m2
                + 500.0
            )
        )
    )

    return float(
        np.clip(
            h,
            HEIGHT_MIN_M,
            HEIGHT_MAX_M
        )
    )


def bearing_deg(
    p_from,
    p_to
):
    dx = (
        p_to.x
        - p_from.x
    )

    dy = (
        p_to.y
        - p_from.y
    )

    ang = np.degrees(
        np.arctan2(
            dx,
            dy
        )
    )

    return (
        ang + 360.0
    ) % 360.0


def ang_diff(
    a,
    b
):
    d = (
        abs(a - b)
        % 360.0
    )

    return min(
        d,
        360.0 - d
    )


def shading_factor_for_point(
    point_xy,
    roof_h,
    neighbors_gdf,
    solpos,
    max_check=50
):
    if len(neighbors_gdf) == 0:
        return pd.Series(
            1.0,
            index=solpos.index
        )

    neigh = neighbors_gdf.copy()

    neigh["dist"] = (
        neigh.geometry.distance(
            point_xy
        )
    )

    neigh = (
        neigh
        .sort_values("dist")
        .head(max_check)
    )

    sun_az = solpos[
        "azimuth"
    ].values

    sun_el = solpos[
        "apparent_elevation"
    ].values

    shaded = np.zeros(
        len(solpos),
        dtype=bool
    )

    for _, row in neigh.iterrows():

        near_pt = (
            row.geometry
            .representative_point()
        )

        bearing = bearing_deg(
            point_xy,
            near_pt
        )

        distance = max(
            float(row["dist"]),
            1.0
        )

        h_obs = building_height_m(
            row.get(
                "area_m2",
                200.0
            )
        )

        dh = max(
            h_obs - roof_h,
            0.0
        )

        obstacle_angle = np.degrees(
            np.arctan2(
                dh,
                distance
            )
        )

        radius = np.sqrt(
            max(
                row.get(
                    "area_m2",
                    100.0
                ),
                10.0
            )
            / np.pi
        )

        half_width = np.degrees(
            np.arctan2(
                radius,
                distance
            )
        )

        dir_match = np.array(
            [
                ang_diff(
                    sa,
                    bearing
                )
                <= half_width
                for sa in sun_az
            ]
        )

        elev_block = (
            sun_el
            <= obstacle_angle
        )

        shaded |= (
            dir_match
            & elev_block
            & (sun_el > 0)
        )

    return pd.Series(
        (~shaded).astype(
            np.float32
        ),
        index=solpos.index
    )

## 14. Estimate capacity, shading, and optimal orientation

In [ ]:
# ============================================================
# 14. PV CAPACITY + ORIENTATION SWEEP
# ============================================================

roof_gdf = gpd.GeoDataFrame(
    {
        "geometry": [roof_poly],
        "area_m2": [
            float(
                b["area_m2"]
            )
        ]
    },
    crs=crs
).to_crs(
    utm_crs
)

neighbors_utm = (
    neighbors
    .to_crs(
        utm_crs
    )
)

if len(neighbors_utm) > 0:
    neighbors_utm[
        "area_m2"
    ] = (
        neighbors_utm
        .geometry
        .area
    )

roof_centroid = (
    roof_gdf
    .geometry
    .iloc[0]
    .centroid
)

roof_area_m2 = float(
    roof_gdf[
        "area_m2"
    ].iloc[0]
)

usable_area = (
    roof_area_m2
    * ROOF_USABLE_FRACTION
)

pdc0_w = (
    usable_area
    * POWER_DENSITY_W_PER_M2
)

roof_h = building_height_m(
    roof_area_m2
)

if USE_SHADING:

    sf = shading_factor_for_point(
        roof_centroid,
        roof_h,
        neighbors_utm,
        solpos,
        max_check=60
    )

else:

    sf = pd.Series(
        1.0,
        index=solpos.index
    )


def annual_energy_kwh(
    tilt,
    azimuth
):
    poa = poa_irradiance(
        tilt,
        azimuth,
        solpos,
        cs
    )

    poa_eff = (
        poa["poa_diffuse"]
        + poa["poa_direct"]
        * sf
    )

    tcell = cell_temperature(
        poa_eff,
        temp_air=20.0,
        wind_speed=1.0
    )

    pdc, pac = pvwatts_power(
        poa_eff,
        tcell,
        pdc0_w
    )

    pac_net = (
        pac
        * (
            1.0
            - SYSTEM_LOSS_FRACTION
        )
    )

    return float(
        pac_net.sum()
        / 1000.0
    )


results = []

for tilt in TILT_GRID:
    for az in AZI_GRID:

        energy = annual_energy_kwh(
            tilt,
            az
        )

        results.append(
            (
                tilt,
                az,
                energy
            )
        )


orientation_df = pd.DataFrame(
    results,
    columns=[
        "tilt_deg",
        "azimuth_deg",
        "annual_kwh"
    ]
)

best = (
    orientation_df
    .sort_values(
        "annual_kwh",
        ascending=False
    )
    .iloc[0]
)

best_tilt = float(
    best["tilt_deg"]
)

best_az = float(
    best["azimuth_deg"]
)

best_kwh = float(
    best["annual_kwh"]
)

kwp = (
    pdc0_w
    / 1000.0
)

specific_yield = (
    best_kwh / kwp
    if kwp > 0
    else np.nan
)

## 15. PV result

In [ ]:
# ============================================================
# 15. SUMMARY + ORIENTATION HEATMAP
# ============================================================

print(
    "====== PV Study "
    "(Clear-sky + simple shading) ======"
)

print(
    f"Building ID: {BID}"
)

print(
    f"Roof/footprint area: "
    f"{roof_area_m2:.1f} m²"
)

print(
    f"Usable PV area: "
    f"{usable_area:.1f} m² "
    f"({ROOF_USABLE_FRACTION*100:.0f}%)"
)

print(
    f"Estimated DC size: "
    f"{kwp:.2f} kWp"
)

print(
    f"Best tilt: "
    f"{best_tilt:.0f}°"
)

print(
    f"Best azimuth: "
    f"{best_az:.0f}° "
    "(0=N, 90=E, 180=S, 270=W)"
)

print(
    f"Annual clear-sky energy estimate: "
    f"{best_kwh:.0f} kWh/year"
)

print(
    f"Specific yield: "
    f"{specific_yield:.0f} "
    "kWh/kWp/year"
)

print(
    f"Generic system losses: "
    f"{SYSTEM_LOSS_FRACTION*100:.0f}%"
)

print(
    f"Simple shading model: "
    f"{'ON' if USE_SHADING else 'OFF'}"
)


pivot = orientation_df.pivot_table(
    index="tilt_deg",
    columns="azimuth_deg",
    values="annual_kwh"
)

plt.figure(
    figsize=(14, 5)
)

plt.imshow(
    pivot.values,
    aspect="auto"
)

plt.xticks(
    range(
        len(
            pivot.columns
        )
    ),
    pivot.columns,
    rotation=90
)

plt.yticks(
    range(
        len(
            pivot.index
        )
    ),
    pivot.index
)

plt.colorbar(
    label="Annual kWh"
)

plt.title(
    "PV annual energy sweep: "
    "tilt vs azimuth"
)

plt.xlabel(
    "Azimuth (degrees)"
)

plt.ylabel(
    "Tilt (degrees)"
)

plt.tight_layout()
plt.show()

## 16. Optional: search for the least-shaded point inside the selected roof

This reproduces the placement-proxy idea from your original notebook. It does **not** model individual modules or real roof obstructions.

In [ ]:
# ============================================================
# 16. BEST PLACEMENT PROXY
# ============================================================

def sample_points_in_polygon(
    poly,
    n=150,
    seed=42
):
    rng = np.random.default_rng(
        seed
    )

    minx, miny, maxx, maxy = (
        poly.bounds
    )

    prepared = prep(
        poly
    )

    pts = []

    tries = 0

    while (
        len(pts) < n
        and tries < n * 100
    ):

        tries += 1

        x = rng.uniform(
            minx,
            maxx
        )

        y = rng.uniform(
            miny,
            maxy
        )

        p = Point(
            x,
            y
        )

        if prepared.contains(p):
            pts.append(p)

    return pts


poly_utm = (
    roof_gdf
    .geometry
    .iloc[0]
)

pts = sample_points_in_polygon(
    poly_utm,
    n=150,
    seed=42
)

if pts:

    scores = []

    for p in pts:

        if USE_SHADING:

            sf_p = (
                shading_factor_for_point(
                    p,
                    roof_h,
                    neighbors_utm,
                    solpos,
                    max_check=60
                )
            )

        else:

            sf_p = pd.Series(
                1.0,
                index=solpos.index
            )

        # Fraction of all timestamps for which
        # the direct-sun proxy is unblocked.
        scores.append(
            float(
                sf_p.mean()
            )
        )

    scores = np.asarray(
        scores
    )

    best_point_idx = int(
        np.argmax(
            scores
        )
    )

    best_point = pts[
        best_point_idx
    ]

    print(
        "Best-point unshaded score:",
        float(
            scores[
                best_point_idx
            ]
        )
    )

    fig, ax = plt.subplots(
        1,
        1,
        figsize=(7, 7)
    )

    gpd.GeoSeries(
        [poly_utm],
        crs=utm_crs
    ).plot(
        ax=ax,
        facecolor="none",
        edgecolor="black"
    )

    sc = ax.scatter(
        [p.x for p in pts],
        [p.y for p in pts],
        s=10,
        c=scores,
        cmap="viridis"
    )

    ax.scatter(
        [best_point.x],
        [best_point.y],
        s=100,
        marker="x"
    )

    plt.colorbar(
        sc,
        ax=ax,
        label="Unshaded score"
    )

    ax.set_title(
        "PV placement proxy"
    )

    ax.set_axis_off()

    plt.show()

else:

    best_point = None
    print(
        "No interior roof points "
        "could be sampled."
    )

## 17. Export the detected buildings and PV result

In [ ]:
# ============================================================
# 17. EXPORT RESULTS
# ============================================================

ALL_BUILDINGS_GEOJSON = os.path.join(
    OUTPUT_DIR,
    "detected_buildings.geojson"
)

PV_BUILDING_GEOJSON = os.path.join(
    OUTPUT_DIR,
    f"pv_building_{BID}.geojson"
)

PV_RESULT_CSV = os.path.join(
    OUTPUT_DIR,
    f"pv_building_{BID}.csv"
)

ORIENTATION_CSV = os.path.join(
    OUTPUT_DIR,
    f"pv_orientation_sweep_{BID}.csv"
)


gdf.to_file(
    ALL_BUILDINGS_GEOJSON,
    driver="GeoJSON"
)


gdf_one = gdf[
    gdf["building_id"]
    == BID
].copy()

gdf_one[
    "roof_area_m2"
] = roof_area_m2

gdf_one[
    "usable_area_m2"
] = usable_area

gdf_one[
    "pdc0_kwp"
] = kwp

gdf_one[
    "best_tilt_deg"
] = best_tilt

gdf_one[
    "best_azimuth_deg"
] = best_az

gdf_one[
    "annual_kwh_clear_sky"
] = best_kwh

gdf_one[
    "specific_yield_kwh_per_kwp"
] = specific_yield

gdf_one.to_file(
    PV_BUILDING_GEOJSON,
    driver="GeoJSON"
)


result_row = {

    "building_id":
        BID,

    "lat":
        lat,

    "lon":
        lon,

    "roof_area_m2":
        roof_area_m2,

    "usable_area_m2":
        usable_area,

    "usable_fraction":
        ROOF_USABLE_FRACTION,

    "pdc0_kwp":
        kwp,

    "best_tilt_deg":
        best_tilt,

    "best_azimuth_deg":
        best_az,

    "annual_kwh_clear_sky":
        best_kwh,

    "specific_yield_kwh_per_kwp":
        specific_yield,

    "system_loss_fraction":
        SYSTEM_LOSS_FRACTION,

    "segmentation_threshold":
        PRED_THRESHOLD,

    "shading_enabled":
        USE_SHADING
}


pd.DataFrame(
    [result_row]
).to_csv(
    PV_RESULT_CSV,
    index=False
)


orientation_df.to_csv(
    ORIENTATION_CSV,
    index=False
)


print("✅ Saved outputs:")

for path in [
    MASK_TIF,
    PROB_TIF,
    ALL_BUILDINGS_GEOJSON,
    PV_BUILDING_GEOJSON,
    PV_RESULT_CSV,
    ORIENTATION_CSV
]:
    print(" -", path)

# What this notebook now gives you

The complete operational chain is now:

```text
Notebook 1
Sentinel-2 → OpenSR → sr.tif

Notebook 2
training dataset → OSM labels → U-Net → unet_buildings.keras

Notebook 3
sr.tif
+ unet_buildings.keras
+ segmentation_training_config.json
        ↓
building_mask.tif
        ↓
detected_buildings.geojson
        ↓
selected building
        ↓
PV capacity + tilt + azimuth + shading proxy
        ↓
annual clear-sky PV estimate
```

The next scientific improvement should be to replace the clear-sky/constant-temperature assumptions with real meteorological data and improve the roof/height/shading representation.